In [1]:
from openfermion import get_fermion_operator

from ofex.clifford import str_tableau
from ofex.state.chem_ref_state import hf_ground, cisd_ground
from ofex.transforms import fermion_to_qubit_state, fermion_to_qubit_operator, find_pauli_symmetry, \
    qubit_reduction_operator
from ofex.utils.chem import molecule_example, run_driver


## 1. Prepare Hamiltonian

For the details of this tutorial, refer to [arXiv:1701.08213](https://arxiv.org/abs/1701.08213).

In [2]:
# Obtain the Fermionic Hamiltonian

mol_name = "H4"

mol = molecule_example(mol_name)
mol = run_driver(mol, run_cisd=True)
fham = mol.get_molecular_hamiltonian()
fham = get_fermion_operator(fham)
f_const = fham.constant
fham = fham - f_const

In [3]:
# Fermion-to-Qubit mapping

transform = "bravyi_kitaev"
f2q_kwargs = {"n_qubits": mol.n_qubits}

n_qubits = mol.n_qubits

hf_state_fermion = hf_ground(mol)
hf_state =fermion_to_qubit_state(hf_state_fermion, transform, **f2q_kwargs)
cisd_state = fermion_to_qubit_state(cisd_ground(mol), transform, **f2q_kwargs)

pham = fermion_to_qubit_operator(fham, transform, **f2q_kwargs)
p_const = pham.constant
# Make qubit hamiltonian traceless
pham = pham - p_const

cisd_energy = mol.cisd_energy - p_const - f_const

## 2. Pauli Symmetry

### 2-1. Find symmetry operators and objects

For a Hamiltonian
$$
    \hat{H} = \sum_j^{N_P} \alpha_j \hat{P}_j,
$$
find set of Pauli symmetry operators $\{\hat{S}_k\}$ such that
$$
    [\hat{S}_k, \hat{P}_j] = 0 \quad \forall k, j \quad \mathrm{thus, }\quad [\hat{H}, \hat{P}_j] = 0.
$$

Symmetry operators are selected with maximal mutual commutativity:
$$
        \mathcal{S}_M=arg\,max_{\mathcal{S}}|\mathcal{S}|\quad \mathrm{s.t. }\quad
        \mathcal{S}\subseteq \{\hat{S}_k: \forall k\}, \quad [\hat{S}_k,
        \hat{S}_l] = 0 \quad \forall \hat{S}_k, \hat{S}_l \in \mathcal{S}.
$$

Furthermore, the symmetry operators are diagonalized as X-type Pauli operators by applying the following Clifford transformation:
$$
    \hat{S}^{(X)}_k = \hat{C} \hat{S}_k \hat{C}^{\dagger} \quad \forall \hat{S}_k \in \mathcal{S}_M.
$$

Then, applying the Clifford transformation to the Hamiltonian enables


In [4]:
symm_operators, clifford_list, reduced_qubits, unitary = find_pauli_symmetry(pham, n_qubits)
print(f"symm_operators =\n\t"+"\n\t".join(str_tableau(symm_operators).split('\n')))
print(f"symm_clifford  = {clifford_list}")
print(f"reduced_qubits = {reduced_qubits}")
print(f"unitary =\n\t" + "\n\t".join(unitary.__str__().split('\n')))


symm_operators =
	1 0 0
	0 1 0
	0 0 1
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	=====
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	0 0 0
	0 0 0
symm_clifford  = ['H_0', 'H_1', 'H_7', 'QSW_2_7', 'CZ_0_4', 'CZ_0_6', 'CZ_0_7', 'CZ_1_5', 'H_0', 'H_1', 'H_2', 'H_0', 'H_1', 'H_2']
reduced_qubits = [0, 1, 2]
unitary =
	0.35355339059327384 [X0 X1 X2] +
	0.35355339059327384 [X0 X1 Z2] +
	0.35355339059327384 [X0 Z1 X2] +
	0.35355339059327384 [X0 Z1 Z2] +
	0.35355339059327384 [Z0 X1 X2] +
	0.35355339059327384 [Z0 X1 Z2] +
	0.35355339059327384 [Z0 Z1 X2] +
	0.35355339059327384 [Z0 Z1 Z2]


In [8]:
qubit_reduced_hamiltonian = qubit_reduction_operator(pham, n_qubits, clifford_list, reduced_qubits, unitary)
print(f"{len(reduced_qubits)} qubits are reduced, thus the original hamiltonian is decomposed to {2 ** len(reduced_qubits)} {n_qubits - len(reduced_qubits)}-qubit Hamiltonians:")
for k, red_op in qubit_reduced_hamiltonian.items():
    print(f"qubit-parity pair: {k}")
    print(f"red_op =\n\t" + '\n\t'.join(red_op.__str__().split('\n')[:3])+"\n\t...")


3 qubits are reduced, thus the original hamiltonian is decomposed to 8 5-qubit Hamiltonians.
qubit-parity pair: ((0, -1), (1, -1), (2, -1))
red_op =
	-0.011706349170573054 [X0 X1 Y2 Y3 Z4] +
	0.026450672123986822 [X0 X1 Y2 Z3 Y4] +
	-0.025863272971678446 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, -1), (2, 1))
red_op =
	0.011706349170573054 [X0 X1 Y2 Y3 Z4] +
	0.026450672123986822 [X0 X1 Y2 Z3 Y4] +
	-0.025863272971678446 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, 1), (2, -1))
red_op =
	-0.011706349170573054 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986822 [X0 X1 Y2 Z3 Y4] +
	-0.025863272971678446 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, -1), (1, 1), (2, 1))
red_op =
	0.011706349170573054 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986822 [X0 X1 Y2 Z3 Y4] +
	-0.025863272971678446 [X0 X1 Z2 X3] +
	...
qubit-parity pair: ((0, 1), (1, -1), (2, -1))
red_op =
	0.011706349170573054 [X0 X1 Y2 Y3 Z4] +
	-0.026450672123986822 [X0 X1 Y2 Z3 Y4] +
	-0.025863272971678446 [X0 X1 Z2 X3] +
	.